Based on Novy-Marx (2012), *The Other Side of Value: The Gross Profitability Premium*

We will verify the availability of the data required to construct and evaluate the **Gross Profitability** signal.

---

### Gross Profitability (GP/A)

Gross Profitability (GP/A), also referred to as Gross Profits-to-Assets, is defined as:


$$
GP/A = \frac{Revenue - Cost\ of\ Goods\ Sold}{Total\ Assets}
     = \frac{REVT - COGS}{AT}
$$


where all variables are obtained from Compustat annual fundamentals.

---

### We will check the following data components

- **Annual accounting data from Compustat**, including revenue (REVT), cost of goods sold (COGS),
  and total assets (AT), which are required to compute Gross Profitability (GP/A)
- **Monthly stock return data from CRSP**, which are used in the paper to compute
  value-weighted portfolio returns
- **The CRSP–Compustat Merged (CCM) linking table**, which enables the connection between
  firm-level accounting data and security-level stock return data
- **Fama–French factor returns (MKT, SMB, HML, MOM)**, which are planned to be incorporated
  in the end-of-quarter analysis for factor-based evaluation of the signal


In addition, we perform basic sanity checks on the constructed GP/A measure and assess whether the accounting data can be successfully linked to CRSP identifiers without structural data issues.


In [1]:
import wrds
import pandas as pd
import numpy as np

In [2]:
# Establish WRDS connection
db = wrds.Connection()

Enter your WRDS username [linx]:lin15
Enter your password:········
WRDS recommends setting up a .pgpass file.
Create .pgpass file now [y/n]?: y
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


**Data Period:**

The results in paper are based on portfolio returns from July 1963 to December 2010, with portfolios formed annually in June
using accounting information from the previous fiscal year.

To avoid boundary issues when aligning accounting information with subsequent stock returns, we extract raw data slightly beyond the main sample period used in the paper, which is from 1961 to 2012.

### Compustat data

In [30]:
# Load Compustat annual data
# Load Compustat annual data (Updated with Book Equity components)
# Load Compustat annual data (Updated with both sic and sich)
# Load Compustat annual data (Joined with comp.company to get header SIC)
compustat_raw = db.raw_sql("""
    SELECT
        f.gvkey,
        f.datadate,
        f.fyear,
        f.revt,
        f.cogs,
        f.at,
        c.sic,               -- Header SIC from comp.company
        f.sich,              -- Historical SIC from comp.funda
        f.seq, f.ceq, f.pstk, f.lt, f.txditc, f.txdb, f.itcb, f.pstkrv, f.pstkl
    FROM comp.funda f
    LEFT JOIN comp.company c 
        ON f.gvkey = c.gvkey
    WHERE f.indfmt = 'INDL'
      AND f.datafmt = 'STD'
      AND f.popsrc = 'D'
      AND f.consol = 'C'
      AND f.fyear BETWEEN 1961 AND 2012
""", date_cols=["datadate"])
print("Compustat raw shape:", compustat_raw.shape)

compustat_raw.head()

Compustat raw shape: (432594, 17)


,gvkey,datadate,fyear,revt,cogs,at,sic,sich,seq,ceq,pstk,lt,txditc,txdb,itcb,pstkrv,pstkl
0,001000,1961-12-31,1961,0.9,<NA>,<NA>,3089,<NA>,<NA>,<NA>,<NA>,<NA>,0.0,0.0,0.0,<NA>,0.0
1,001002,1961-12-31,1961,8.0,<NA>,<NA>,3825,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.0
2,001187,1961-12-31,1961,0.975,0.991,0.946,1000,<NA>,0.591,0.591,0.0,<NA>,<NA>,<NA>,0.0,0.0,0.0
3,001040,1961-12-31,1961,516.5,382.58,366.6,3949,<NA>,<NA>,<NA>,<NA>,212.746,13.701,13.701,<NA>,7.459,7.2
4,001010,1962-04-30,1961,214.3,168.57,251.2,3743,<NA>,<NA>,<NA>,<NA>,128.316,11.562,11.562,<NA>,0.0,0.0


In [31]:
# Check missing rate of accounting variables in Compustat required for GP/A
compustat_missing_rate = compustat_raw[['revt', 'cogs', 'at']].isnull().mean()
compustat_missing_rate

revt    0.118007
cogs    0.123573
at      0.115388
dtype: float64

The missing rates for the Compustat accounting variables required to construct GP/A are approximately 11–12% for REVT, COGS, and AT. This is expected, as not all firms report complete accounting information in every fiscal year, especially in earlier sample periods. In the final project, GP/A will be computed only for firm-year observations with non-missing inputs, following standard practice in the asset pricing literature.

In [32]:
# Save raw compustat data
compustat_raw.to_parquet("raw_compustat_funda_1961_2012.parquet", index=False)

### CRSP data

In [7]:
# Load CRSP monthly stock returns data
crsp_raw = db.raw_sql("""
    SELECT
        permno,
        permco,
        date,
        ret,
        retx,
        prc,
        shrout
    FROM crsp.msf
    WHERE date BETWEEN '1961-01-01' AND '2012-12-31'
""", date_cols=["date"])

print("CRSP raw shape:", crsp_raw.shape)

crsp_raw.head()

CRSP raw shape: (3628788, 7)


,permno,permco,date,ret,retx,prc,shrout
0,10000,7952,1985-12-31,<NA>,<NA>,<NA>,<NA>
1,10000,7952,1986-01-31,<NA>,<NA>,-4.375,3680.0
2,10000,7952,1986-02-28,-0.257143,-0.257143,-3.25,3680.0
3,10000,7952,1986-03-31,0.365385,0.365385,-4.4375,3680.0
4,10000,7952,1986-04-30,-0.098592,-0.098592,-4.0,3793.0


In [9]:
# This should output '1961-01-31' based on your SQL WHERE clause
print(crsp_raw['date'].min()) 

1961-01-31 00:00:00


In [10]:
# Check missing rate
crsp_missing_rate = crsp_raw[['ret', 'prc', 'shrout']].isnull().mean()
crsp_missing_rate

ret       0.042553
prc       0.034405
shrout    0.007410
dtype: float64

The missing rates for CRSP return and price variables are low, indicating that the market data are sufficiently complete for portfolio construction.

Missing values in key variables will be excluded during signal construction and portfolio analysis.

In [11]:
crsp_raw['date'] = pd.to_datetime(crsp_raw['date'])

# Market equity
crsp_raw['me'] = abs(crsp_raw['prc']) * crsp_raw['shrout']

In [12]:
msenames = db.raw_sql("""
    SELECT
        permno,
        namedt,
        nameendt,
        shrcd,
        exchcd
    FROM crsp.msenames
""")

msenames['namedt'] = pd.to_datetime(msenames['namedt'])
msenames['nameendt'] = pd.to_datetime(msenames['nameendt'])

msenames.head()

,permno,namedt,nameendt,shrcd,exchcd
0,10000,1986-01-07,1986-12-03,10,3
1,10000,1986-12-04,1987-03-09,10,3
2,10000,1987-03-10,1987-06-11,10,3
3,10001,1986-01-09,1993-11-21,11,3
4,10001,1993-11-22,2004-06-09,11,3


In [13]:
msenames.shape

(117830, 5)

In [14]:
crsp = crsp_raw.merge(msenames, on='permno', how='left')

crsp = crsp[
    (crsp['date'] >= crsp['namedt']) &
    (crsp['date'] <= crsp['nameendt'])
]

In [15]:
crsp.shape

(3582225, 12)

In [16]:
# Common shares only
crsp = crsp[crsp['shrcd'].isin([10, 11])]

# Major exchanges only (NYSE=1, AMEX=2, NASDAQ=3)
crsp = crsp[crsp['exchcd'].isin([1, 2, 3])]

crsp.shape

(2904107, 12)

In [17]:
crsp['date'].min()

Timestamp('1961-01-31 00:00:00')

In [18]:
# Save filtered CRSP data
crsp.to_parquet("crsp_msf_msenames_filtered_1961_2012.parquet", index=False)

### (New) Delisting Returns

CRSP Delisting Returns (CRITICAL)

Most students forget this.

Without delisting returns:

You overstate strategy performance

You introduce survivorship bias

Later we merge dlret into CRSP monthly returns.

In [19]:
dlret = db.raw_sql("""
    SELECT
        permno,
        dlret,
        dlstdt
    FROM crsp.msedelist
""")

dlret['dlstdt'] = pd.to_datetime(dlret['dlstdt'])

dlret.head()

,permno,dlret,dlstdt
0,10000,0.0,1987-06-11
1,10001,0.011583,2017-08-03
2,10002,0.046007,2013-02-15
3,10003,0.01373,1995-12-15
4,10005,0.125,1991-07-11


In [20]:
dlret.to_parquet("msedelist.parquet", index=False)

### CRSP-Compustat Merged link table

In [21]:
# Load CRSP–Compustat Merged (CCM) link table
ccm_raw = db.raw_sql("""
    SELECT
        gvkey,
        lpermno AS permno,
        liid AS iid,
        linktype,
        linkprim,
        linkdt,
        linkenddt
    FROM crsp.ccmxpf_lnkhist
    WHERE linktype IN ('LU', 'LC')   -- Keep reliable links only
      AND linkprim IN ('P', 'C')     -- Keep primary links
""", date_cols=["linkdt", "linkenddt"])

ccm_raw['linkdt'] = pd.to_datetime(ccm_raw['linkdt'])
ccm_raw['linkenddt'] = pd.to_datetime(ccm_raw['linkenddt'])


print("CCM raw shape:", ccm_raw.shape)

ccm_raw.head()

CCM raw shape: (33324, 7)


,gvkey,permno,iid,linktype,linkprim,linkdt,linkenddt
0,001000,25881.0,01,LU,P,1970-11-13,1978-06-30
1,001001,10015.0,01,LU,P,1983-09-20,1986-07-31
2,001002,10023.0,01,LC,C,1972-12-14,1973-06-05
3,001003,10031.0,01,LU,C,1983-12-07,1989-08-16
4,001004,54594.0,01,LU,P,1972-04-24,NaT


In [22]:
# Save raw CCM data
ccm_raw.to_parquet("raw_ccm_linktable.parquet", index=False)

In [17]:
# Merge Compustat with CCM to evaluate link feasibility
comp_ccm = compustat_raw.merge(ccm_raw, on='gvkey', how='left')
comp_ccm.head()

,gvkey,datadate,fyear,revt,cogs,at,permno,iid,linktype,linkprim,linkdt,linkenddt
0,001000,1961-12-31,1961,0.9,<NA>,<NA>,25881.0,01,LU,P,1970-11-13,1978-06-30
1,001002,1961-12-31,1961,8.0,<NA>,<NA>,10023.0,01,LC,C,1972-12-14,1973-06-05
2,001187,1961-12-31,1961,0.975,0.991,0.946,<NA>,<NA>,<NA>,<NA>,NaT,NaT
3,001040,1961-12-31,1961,516.5,382.58,366.6,15763.0,00X,LU,C,1950-01-01,1962-01-30
4,001040,1961-12-31,1961,516.5,382.58,366.6,15763.0,01,LU,P,1962-01-31,1985-10-31


In [18]:
# Calculate merge success rate
merge_rate = comp_ccm['permno'].notnull().mean()
print(f"CCM merge success rate: {merge_rate:.2%}")

CCM merge success rate: 83.63%


The merge success rate indicates that a large majority of Compustat firm-year observations can be linked to CRSP securities using the CCM link table, suggesting that the data required to construct and test the GP/A signal is broadly available.

**Summary:**

Overall, we check that all required data sources for constructing GP/A and linking to stock returns are accessible. No data availability constraints are identified that would prevent replication of the paper’s signal in the final project.

### Fama–French factor data
**Use to evaluate signal in end-quarter presentation**

In [23]:
# Load Fama–French monthly factor data
ff_factors_raw = db.raw_sql("""
    SELECT
        date,
        mktrf,
        smb,
        hml,
        umd,
        rf
    FROM ff.factors_monthly
    WHERE date BETWEEN '1961-01-01' AND '2012-12-31'
""", date_cols=["date"])

print("FF factors shape:", ff_factors_raw.shape)

ff_factors_raw.head()


FF factors shape: (624, 6)


,date,mktrf,smb,hml,umd,rf
0,1961-01-01,0.062,0.006,0.0365,-0.0412,0.0019
1,1961-02-01,0.0357,0.0389,-0.0058,0.0099,0.0014
2,1961-03-01,0.0289,0.0322,-0.0082,0.0432,0.002
3,1961-04-01,0.0029,0.0007,0.0213,0.0362,0.0017
4,1961-05-01,0.024,0.0197,0.0038,-0.0157,0.0018


In [24]:
ff_factors_raw.isnull().sum()

date     0
mktrf    0
smb      0
hml      0
umd      0
rf       0
dtype: int64

In [25]:
# Save raw data
ff_factors_raw.to_parquet("raw_ff_factors_monthly_1961_2012.parquet", index=False)

### WRDS Signal (GP/A)

In [26]:
# Download WRDS pre-built GP/A signal
query_gpa = """
    SELECT fdate, permno, gprof
    FROM wrdsapps_backtest_plus.signals_raw_plus
"""
wrds_gpa = db.raw_sql(query_gpa)

wrds_gpa.to_parquet('wrds_gpa_signal.parquet', index=False)
print("Shape:", wrds_gpa.shape)
print(wrds_gpa.head())

Shape: (3417553, 3)
        fdate  permno     gprof
0  1986-06-30   10000  0.009177
1  1986-07-31   10000  0.009177
2  1986-08-31   10000  0.009177
3  1986-09-30   10000   0.01048
4  1986-10-31   10000   0.01048


In [28]:
wrds_gpa.sort_values(by='fdate')

,fdate,permno,gprof
61166,1960-01-31,18315,0.221027
38730,1960-01-31,17865,0.184214
336391,1960-01-31,13805,<NA>
312558,1960-01-31,13311,0.689695
59465,1960-01-31,18278,0.424714
...,...,...,...
369595,2024-12-31,77202,-0.077375
488519,2024-12-31,16648,0.150745
488610,2024-12-31,16649,0.223425
489128,2024-12-31,16653,0.239605
